# Parte 1 — Análise Exploratória de Dados (EDA)

Objetivo: entender os dados antes de modelar e responder às perguntas da Parte 1 do desafio.
As justificativas de cada decisão estão no [README](../README.md#parte-1--análise-exploratória).

Roteiro: 1. Contexto 
· 2. Divisão treino/teste 
· 3. Formato e tipos 
· 4. Integridade 
· 5. Distribuição de cada variável 
· 6. Relação entre pares de variáveis 
· 7. Efeito de uma terceira variável 
· 8. Registro dos achados

## Passo 1 — Contexto

- Linha: uma campanha (um vídeo veiculado em uma plataforma).
- Alvo: `klike_score` (0 a 100), qualidade geral do criativo segundo a IA da Klike.
- Esperado: 500 registros e 26 colunas.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/klike_challenge_dataset.csv")
df.shape

(500, 26)

## Passo 2 — Divisão treino/teste

Separamos o teste antes de explorar, para que nenhuma decisão seja influenciada por ele (evitar *data snooping*).
Daqui em diante, toda a análise usa só o `train`.

- 80% treino (400) / 20% teste (100)
- `random_state=42`: divisão reprodutível
- `stratify=platform`: mesmas proporções de plataforma nos dois conjuntos

In [2]:
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["platform"])
len(train), len(test)

(400, 100)

In [3]:
proporcoes = pd.DataFrame({
    "base": df["platform"].value_counts(normalize=True),
    "treino": train["platform"].value_counts(normalize=True),
    "teste": test["platform"].value_counts(normalize=True),
})
(proporcoes * 100).round(1)

,base,treino,teste
platform,,,
Meta,48.2,48.2,48.0
TikTok,35.2,35.2,35.0
LinkedIn,16.6,16.5,17.0


Proporções preservadas: a maior diferença em relação à base é de 0,4 ponto percentual (LinkedIn).

## Passo 3 — Formato e tipos

Conferir se cada coluna foi lida com o tipo certo e quais categorias existem. Só observamos aqui; as conversões de tipo ficam para o pré-processamento.

In [4]:
exemplos = train.head().T
exemplos.columns = [f"Exemplo {i}" for i in range(1, 6)]
exemplos

,Exemplo 1,Exemplo 2,Exemplo 3,Exemplo 4,Exemplo 5
campaign_id,KLK-0057,KLK-0301,KLK-0088,KLK-0230,KLK-0485
date,2024-02-07,2024-02-10,2024-10-24,2024-05-05,2024-01-05
platform,Meta,Meta,TikTok,TikTok,TikTok
category,E-commerce,Branding,E-commerce,E-commerce,Branding
objective,conversions,awareness,traffic,traffic,engagement
target_audience_age,18-24,35-44,18-24,18-24,18-24
is_retargeting,False,False,False,False,True
video_duration_s,13.0,11.0,11.0,5.0,11.0
format,quadrado,vertical,vertical,vertical,vertical
has_subtitle,True,True,True,False,False


In [5]:
tipos = pd.DataFrame({
    "tipo": train.dtypes.astype(str),
    "preenchidos": train.notna().sum(),
    "faltantes": train.isna().sum(),
    "exemplo": train.apply(lambda col: col.dropna().iloc[0]),
})
tipos

,tipo,preenchidos,faltantes,exemplo
campaign_id,object,400,0,KLK-0057
date,object,400,0,2024-02-07
platform,object,400,0,Meta
category,object,400,0,E-commerce
objective,object,400,0,conversions
target_audience_age,object,400,0,18-24
is_retargeting,bool,400,0,False
video_duration_s,float64,370,30,13.0
format,object,400,0,quadrado
has_subtitle,object,364,36,True


Verificação das categorias contra o dicionário de dados do desafio. Uma categoria nova não é tratada como erro, apenas sinalizada como "fora do dicionário" (ver README, Passo 3).

In [6]:
# Categorias esperadas segundo o dicionário de dados do desafio
CATEGORIAS_ESPERADAS = {
    "platform": ["Meta", "TikTok", "LinkedIn"],
    "category": ["E-commerce", "SaaS", "App Install", "Branding", "Lead Gen"],
    "objective": ["awareness", "traffic", "conversions", "engagement", "app_install"],
    "target_audience_age": ["18-24", "25-34", "35-44", "45+"],
    "format": ["vertical", "horizontal", "quadrado"],
    "text_density": ["low", "medium", "high"],
}


def checar_categorias(col):
    encontradas = set(train[col].dropna().unique())
    esperadas = set(CATEGORIAS_ESPERADAS[col])
    problemas = []
    if encontradas - esperadas:
        problemas.append("fora do dicionário: " + ", ".join(sorted(encontradas - esperadas)))
    if esperadas - encontradas:
        problemas.append("ausentes: " + ", ".join(sorted(esperadas - encontradas)))
    return "; ".join(problemas) or "ok"


pd.DataFrame(
    {"resultado": [checar_categorias(col) for col in CATEGORIAS_ESPERADAS]},
    index=pd.Index(list(CATEGORIAS_ESPERADAS), name="coluna"),
)

,resultado
coluna,
platform,ok
category,ok
objective,ok
target_audience_age,ok
format,ok
text_density,ok


Observações:
- Tipos a corrigir no pré-processamento: `date` (texto → data), `has_subtitle` (texto por causa de 36 vazios → verdadeiro/falso), `text_density` e `target_audience_age` (categorias com ordem natural, hoje sem ordem).
- Categorias consistentes com o dicionário de dados, sem variações de escrita.
- 7 colunas com valores faltantes (20 a 36 cada), analisadas no Passo 4.

## Passo 4 — Integridade

### 4.1 Valores faltantes

Quantos valores faltam, onde, e se faltam ao acaso ou ligados a outras características da campanha.

In [7]:
colunas_com_faltantes = train.columns[train.isna().any()].tolist()

faltantes = pd.DataFrame({
    "faltantes": train[colunas_com_faltantes].isna().sum(),
    "% do total": train[colunas_com_faltantes].isna().mean() * 100,
})
faltantes.style.format({"% do total": "{:.1f}%"})

,faltantes,% do total
video_duration_s,30,7.5%
has_subtitle,36,9.0%
music_voice_ratio,29,7.2%
cpc,20,5.0%
revenue,22,5.5%
avg_watch_time_s,22,5.5%
engagement_rate,27,6.8%


Entre 5% e 9% de vazios por coluna: pouco o bastante para nenhuma coluna precisar ser descartada.

In [8]:
vazios_por_campanha = train.isna().sum(axis=1).value_counts().sort_index()

tabela_vazios = pd.DataFrame(
    {
        "campanhas": vazios_por_campanha.values,
        "% do total": vazios_por_campanha.values / len(train) * 100,
    },
    index=pd.Index(["nenhum" if n == 0 else str(n) for n in vazios_por_campanha.index],
                   name="campos vazios na campanha"),
)
tabela_vazios.style.format({"% do total": "{:.1f}%"})

,campanhas,% do total
campos vazios na campanha,,
nenhum,250,62.5%
1,118,29.5%
2,28,7.0%
3,4,1.0%


62,5% das campanhas estão completas e quase todas as demais têm um único campo vazio. Os vazios estão espalhados, e não concentrados em poucas campanhas problemáticas.

In [9]:
# % de faltantes de cada coluna dentro de cada plataforma
(train[colunas_com_faltantes].isna().groupby(train["platform"]).mean() * 100).style.format("{:.1f}%")

,video_duration_s,has_subtitle,music_voice_ratio,cpc,revenue,avg_watch_time_s,engagement_rate
platform,,,,,,,
LinkedIn,9.1%,9.1%,12.1%,3.0%,6.1%,1.5%,7.6%
Meta,5.7%,8.3%,6.2%,5.2%,4.7%,5.2%,6.7%
TikTok,9.2%,9.9%,6.4%,5.7%,6.4%,7.8%,6.4%


Taxas parecidas entre plataformas. A maior (`music_voice_ratio` no LinkedIn, 12,1%) corresponde a apenas 8 campanhas.

Teste: para cada coluna com faltantes, comparamos as campanhas com e sem o valor vazio nas demais variáveis.
Categóricas: teste qui-quadrado. Numéricas: teste de Mann-Whitney. Como são muitas comparações, os p-valores são corrigidos pelo método de Holm. Listamos as diferenças com p < 0,05 antes da correção.

In [10]:
import numpy as np
from scipy.stats import chi2_contingency, mannwhitneyu

VARIAVEIS_CATEGORICAS = ["platform", "category", "objective", "target_audience_age", "format",
                         "text_density", "is_retargeting", "has_hook", "has_face", "has_cta"]
VARIAVEIS_NUMERICAS = ["klike_score", "impressions", "spend", "ctr", "video_duration_s",
                       "engagement_rate", "roas"]


def comparar_com_e_sem_faltante(dados, col):
    vazio = dados[col].isna()
    resultados = []
    for var in VARIAVEIS_CATEGORICAS:
        if var != col:
            p = chi2_contingency(pd.crosstab(dados[var], vazio))[1]
            resultados.append({"variável": var, "p": p, "mediana com vazio": None, "mediana sem vazio": None})
    for var in VARIAVEIS_NUMERICAS:
        if var != col:
            com, sem = dados.loc[vazio, var].dropna(), dados.loc[~vazio, var].dropna()
            resultados.append({"variável": var, "p": mannwhitneyu(com, sem).pvalue,
                               "mediana com vazio": com.median(), "mediana sem vazio": sem.median()})
    return pd.DataFrame(resultados).assign(coluna=col)


def corrigir_holm(p):
    """p-valores ajustados pelo método de Holm para comparações múltiplas."""
    ordem = p.sort_values()
    ajustado = (ordem * (len(ordem) - np.arange(len(ordem)))).cummax().clip(upper=1)
    return ajustado.reindex(p.index)


testes = pd.concat([comparar_com_e_sem_faltante(train, col) for col in colunas_com_faltantes], ignore_index=True)
testes["p corrigido"] = corrigir_holm(testes["p"])
print(f"Total de comparações: {len(testes)}")
testes.loc[testes["p"] < 0.05,
           ["coluna", "variável", "p", "p corrigido", "mediana com vazio", "mediana sem vazio"]].round(3)

Total de comparações: 117


,coluna,variável,p,p corrigido,mediana com vazio,mediana sem vazio
62,cpc,spend,0.048,1.000,2387.335,966.255
63,cpc,ctr,0.042,1.000,0.101,0.063
95,avg_watch_time_s,impressions,0.002,0.267,9045.000,25331.000
96,avg_watch_time_s,spend,0.003,0.391,354.015,1107.300


Das 117 comparações, 4 têm p < 0,05, e nenhuma continua significativa após a correção de Holm: não encontramos evidência contra MCAR. Ainda assim, `avg_watch_time_s` tem os menores p-valores e medianas de impressões e gasto cerca de três vezes menores nas campanhas com vazio. Olhamos esse caso mais de perto abaixo.

In [11]:
# % de campanhas sem avg_watch_time_s em cada faixa de impressões (5 faixas com 80 campanhas cada)
faixa_impressoes = pd.qcut(train["impressions"], 5,
                           labels=["1 (menores)", "2", "3", "4", "5 (maiores)"])
vazio_watch = train["avg_watch_time_s"].isna()

pd.DataFrame({
    "campanhas sem o valor": vazio_watch.groupby(faixa_impressoes, observed=True).sum(),
    "% da faixa": vazio_watch.groupby(faixa_impressoes, observed=True).mean() * 100,
}).rename_axis("faixa de impressões").style.format({"% da faixa": "{:.1f}%"})

,campanhas sem o valor,% da faixa
faixa de impressões,,
1 (menores),11,13.8%
2,3,3.8%
3,3,3.8%
4,4,5.0%
5 (maiores),1,1.2%


Metade dos vazios de `avg_watch_time_s` (11 de 22) está nos 20% de campanhas com menos impressões. Uma explicação plausível: com poucas visualizações, a plataforma não reporta o tempo médio assistido.

Observações do 4.1:
- Nenhuma diferença sobrevive à correção de Holm: sem evidência contra MCAR nas 7 colunas.
- Exceção provável: `avg_watch_time_s` falta mais em campanhas pequenas (poucas impressões), o que indica MAR. O tratamento precisa levar isso em conta.
- O que fazer com os vazios fica para o pré-processamento (ver README, Passo 4.1).
- `cpc` e `revenue` talvez possam ser recalculados a partir de outras colunas (`spend / clicks` e `roas × spend`); isso é verificado no 4.3.

### 4.2 Duplicatas

Procuramos linhas repetidas em quatro níveis, do mais estrito ao mais amplo, e campanhas presentes no treino e no teste ao mesmo tempo. A última checagem só compara se as linhas são iguais; nenhuma estatística do teste é calculada.

In [12]:
# o que define "a mesma campanha", sem as métricas
CONTEXTO_E_CRIATIVO = ["platform", "category", "objective", "target_audience_age", "is_retargeting",
                       "video_duration_s", "format", "has_subtitle", "has_cta", "has_hook", "has_face",
                       "text_density", "music_voice_ratio"]

sem_id = train.columns.drop("campaign_id")

pd.DataFrame(
    {"linhas": [
        train.duplicated().sum(),
        train["campaign_id"].duplicated().sum(),
        train.duplicated(subset=sem_id).sum(),
        train.duplicated(subset=CONTEXTO_E_CRIATIVO).sum(),
        len(train[sem_id].merge(test[sem_id])),
    ]},
    index=pd.Index([
        "linhas idênticas no treino",
        "campaign_id repetido no treino",
        "linhas idênticas ignorando o campaign_id",
        "mesmo contexto e mesmo vídeo, métricas diferentes",
        "campanhas no treino e no teste ao mesmo tempo",
    ], name="verificação"),
)

,linhas
verificação,
linhas idênticas no treino,0
campaign_id repetido no treino,0
linhas idênticas ignorando o campaign_id,0
"mesmo contexto e mesmo vídeo, métricas diferentes",0
campanhas no treino e no teste ao mesmo tempo,0


Nenhuma duplicata em nenhum nível, e nenhuma sobreposição entre treino e teste: não há linhas a remover.

### 4.3 Valores impossíveis e inconsistentes

Primeiro, regras que valem por definição para qualquer campanha.

In [13]:
pd.DataFrame(
    {"campanhas que violam": [
        (train.select_dtypes("number") < 0).any(axis=1).sum(),
        (train["clicks"] > train["impressions"]).sum(),
        (train["conversions"] > train["clicks"]).sum(),
        (train["avg_watch_time_s"] > train["video_duration_s"]).sum(),
        (~train[["ctr", "engagement_rate", "music_voice_ratio"]].stack().between(0, 1)).sum(),
        (~train["klike_score"].between(0, 100)).sum(),
        (~pd.to_datetime(train["date"]).dt.year.eq(2024)).sum(),
    ]},
    index=pd.Index([
        "algum valor negativo",
        "mais cliques que impressões",
        "mais conversões que cliques",
        "tempo assistido maior que a duração do vídeo",
        "taxa fora de 0 a 1 (ctr, engagement_rate, music_voice_ratio)",
        "klike_score fora de 0 a 100",
        "data fora de 2024",
    ], name="regra"),
)

,campanhas que violam
regra,
algum valor negativo,0
mais cliques que impressões,0
mais conversões que cliques,0
tempo assistido maior que a duração do vídeo,0
"taxa fora de 0 a 1 (ctr, engagement_rate, music_voice_ratio)",0
klike_score fora de 0 a 100,0
data fora de 2024,0


Nenhuma violação.

As métricas derivadas também precisam fechar com as colunas de origem: `ctr = clicks / impressions`, `cpc = spend / clicks` e `roas = revenue / spend`. Medimos a diferença relativa entre o valor registrado e o recalculado.

In [14]:
diferencas = pd.DataFrame({
    "ctr": train["ctr"] / (train["clicks"] / train["impressions"]) - 1,
    "cpc": train["cpc"] / (train["spend"] / train["clicks"]) - 1,
    "roas": train["roas"] / (train["revenue"] / train["spend"]) - 1,
}).abs()

faixas = pd.cut(diferencas.stack(), [0, 0.01, 0.1, 0.5, float("inf")], include_lowest=True,
                labels=["até 1%", "1% a 10%", "10% a 50%", "acima de 50%"])
faixas.groupby([faixas.index.get_level_values(1), faixas], observed=False).size().unstack()

,até 1%,1% a 10%,10% a 50%,acima de 50%
cpc,374,0,0,6
ctr,385,11,0,4
roas,302,19,0,2


A distribuição tem um vão: ou a diferença é de até 10% (arredondamento das métricas, gravadas com 2 a 4 casas decimais), ou passa de 70%. Nada fica entre os dois. Tratamos acima de 20% como conta quebrada.

Quando uma conta quebra, uma das duas colunas envolvidas está errada. A regra de correção fica em `src/limpeza.py`, para que a EDA e o pré-processamento usem exatamente a mesma lógica:
- `ctr` não fecha: o erro está em `impressions` ou no próprio `ctr`. Fica o `ctr` mais plausível para a plataforma, e a outra coluna é recalculada.
- `cpc` não fecha: mesmo raciocínio entre `spend` e `cpc`.
- "Mais plausível" = mais perto da mediana da plataforma no treino, em escala log.
- A coluna `coluna_corrigida` registra o que foi alterado.

In [15]:
import sys
sys.path.append("..")
from src.limpeza import corrigir_inconsistencias, medianas_por_plataforma

train_corrigido = corrigir_inconsistencias(train, medianas_por_plataforma(train))
corrigidas = train_corrigido["coluna_corrigida"].notna()

comparacao = pd.DataFrame({
    "campanha": train.loc[corrigidas, "campaign_id"],
    "coluna corrigida": train_corrigido.loc[corrigidas, "coluna_corrigida"],
})
for col in ["impressions", "cpc", "spend"]:
    alterada = comparacao["coluna corrigida"] == col
    comparacao.loc[alterada, "antes"] = train.loc[corrigidas, col][alterada]
    comparacao.loc[alterada, "depois"] = train_corrigido.loc[corrigidas, col][alterada]
comparacao.sort_values("coluna corrigida").set_index("campanha")

,coluna corrigida,antes,depois
campanha,,,
KLK-0240,cpc,24.17,0.98
KLK-0100,cpc,20.00,0.39
KLK-0416,cpc,26.05,0.74
KLK-0232,impressions,12594257.00,11695.00
KLK-0258,impressions,12873562.00,86948.00
KLK-0413,impressions,14052761.00,72059.00
KLK-0148,impressions,13007321.00,42500.00
KLK-0259,spend,95811.69,5.40
KLK-0224,spend,85023.80,21557.40


In [16]:
# o roas não entra na regra, então serve de validação
c = train_corrigido[corrigidas]
fecha = (c["roas"] - c["revenue"] / c["spend"]).abs() < 0.01
print(f"roas fecha em {fecha.sum()} de {len(c)} campanhas corrigidas")

roas fecha em 10 de 10 campanhas corrigidas


10 campanhas (2,5% do treino) têm exatamente uma coluna com erro de registro, e o valor correto pode ser recuperado pelas outras colunas:
- 4 com `impressions` na casa de 13 milhões: o `ctr` e os cliques indicam entre 12 mil e 87 mil.
- 3 com `cpc` entre 20 e 26: `spend / clicks` dá entre 0,39 e 0,98.
- 3 com `spend` inflado: `cpc × clicks` recupera o gasto (ex: KLK-0259, de 95.811 para 5,40).

O `roas` fecha em todas depois da correção, o que confirma qual coluna estava errada.

Receita zero com conversões registradas:

In [17]:
receita_zero = train["revenue"] == 0
pd.DataFrame({
    "campanhas": train.groupby("objective").size(),
    "receita zero": receita_zero.groupby(train["objective"]).sum(),
    "receita zero com conversões": (receita_zero & (train["conversions"] > 0)).groupby(train["objective"]).sum(),
})

,campanhas,receita zero,receita zero com conversões
objective,,,
app_install,37,0,0
awareness,40,22,19
conversions,129,3,0
engagement,66,27,22
traffic,128,3,0


In [18]:
pd.DataFrame({
    "campanhas": train.groupby("category").size(),
    "receita zero": receita_zero.groupby(train["category"]).sum(),
})

,campanhas,receita zero
category,,
App Install,68,0
Branding,49,47
E-commerce,137,1
Lead Gen,73,2
SaaS,73,5


As campanhas com receita zero e conversões são todas de awareness ou engagement, e 47 das 49 campanhas de Branding têm receita zero. Nesses casos, receita zero indica receita não medida, e não performance ruim: o `roas` dessas campanhas não mede retorno.

Vazios de `cpc` e `revenue` (ver 4.1): verificamos se as colunas necessárias para recalculá-los estão disponíveis.

In [19]:
pd.DataFrame({
    "vazios": [train["cpc"].isna().sum(), train["revenue"].isna().sum()],
    "recalculáveis": [
        (train["cpc"].isna() & (train["clicks"] > 0) & train["spend"].notna()).sum(),
        (train["revenue"].isna() & train["roas"].notna() & train["spend"].notna()).sum(),
    ],
}, index=pd.Index(["cpc = spend / clicks", "revenue = roas × spend"], name="coluna"))

,vazios,recalculáveis
coluna,,
cpc = spend / clicks,20,20
revenue = roas × spend,22,22


Todos os vazios das duas colunas podem ser recalculados. O `cpc` sai exato; a `revenue`, com o erro de arredondamento do `roas` (2 casas decimais).

Os testes de faltantes do 4.1 usaram os dados com os 10 erros. Repetimos os testes com os valores corrigidos para checar se a conclusão muda.

In [20]:
testes_corrigidos = pd.concat([comparar_com_e_sem_faltante(train_corrigido, col) for col in colunas_com_faltantes],
                              ignore_index=True)
testes_corrigidos["p corrigido"] = corrigir_holm(testes_corrigidos["p"])
testes_corrigidos.loc[testes_corrigidos["p"] < 0.05,
                      ["coluna", "variável", "p", "p corrigido", "mediana com vazio", "mediana sem vazio"]].round(3)

,coluna,variável,p,p corrigido,mediana com vazio,mediana sem vazio
62,cpc,spend,0.040,1.000,2387.335,930.955
63,cpc,ctr,0.042,1.000,0.101,0.063
95,avg_watch_time_s,impressions,0.003,0.293,9045.000,25056.500
96,avg_watch_time_s,spend,0.004,0.486,354.015,1058.520


Com os valores corrigidos, as mesmas 4 comparações têm p < 0,05 e nenhuma sobrevive à correção de Holm: a conclusão do 4.1 não muda.

Observações do 4.3:
- Nenhuma violação das regras que valem por definição.
- 10 campanhas com erro de registro em uma única coluna (`impressions`, `cpc` ou `spend`), recuperável pelas demais. A correção será aplicada no pré-processamento com `src/limpeza.py`.
- Receita zero em awareness, engagement e Branding indica receita não medida: o `roas` dessas campanhas não mede retorno.
- Os vazios de `cpc` e `revenue` podem ser recalculados em vez de estimados.
- A conclusão sobre os faltantes (4.1) se mantém com os dados corrigidos.

### 4.4 Tetos e pisos

Valores acumulados exatamente no mínimo ou no máximo de uma coluna indicam truncamento: o valor real foi substituído pelo limite. Usamos o treino já corrigido no 4.3, para que os erros de registro não se confundam com limites.

In [21]:
colunas_numericas = train_corrigido.select_dtypes("number").columns
pd.DataFrame({
    "mínimo": train_corrigido[colunas_numericas].min(),
    "campanhas no mínimo": train_corrigido[colunas_numericas].eq(train_corrigido[colunas_numericas].min()).sum(),
    "máximo": train_corrigido[colunas_numericas].max(),
    "campanhas no máximo": train_corrigido[colunas_numericas].eq(train_corrigido[colunas_numericas].max()).sum(),
})

,mínimo,campanhas no mínimo,máximo,campanhas no máximo
video_duration_s,5.0000,16,115.00,1
music_voice_ratio,0.0200,1,0.92,1
impressions,1000.0000,2,1570242.00,1
clicks,4.0000,1,72545.00,1
ctr,0.0022,1,0.15,42
cpc,0.1500,1,3.94,1
spend,2.9600,1,84152.20,1
conversions,0.0000,16,3666.00,1
revenue,0.0000,55,241597.29,1
roas,0.0000,59,27.42,1


Acúmulos que chamam atenção: 42 campanhas com `ctr` exatamente 0,15 e 7 com `engagement_rate` exatamente 0,10. Os zeros de `conversions`, `revenue` e `roas` são naturais (ver 4.3). Os pisos de `impressions` (1.000) e `video_duration_s` (5 s) são analisados no fim.

In [22]:
faixas_ctr = pd.cut(train_corrigido["ctr"], [0.09, 0.11, 0.13, 0.1499, 0.15],
                    labels=["0,09 a 0,11", "0,11 a 0,13", "0,13 a 0,15", "exatamente 0,15"])
faixas_ctr.value_counts().sort_index().rename("campanhas").to_frame()

,campanhas
ctr,
"0,09 a 0,11",40
"0,11 a 0,13",26
"0,13 a 0,15",24
"exatamente 0,15",42


In [23]:
no_teto_ctr = train_corrigido["ctr"] == 0.15
(train_corrigido.loc[no_teto_ctr, "clicks"] / train_corrigido.loc[no_teto_ctr, "impressions"]).round(3).value_counts()

0.15    42
Name: count, dtype: int64

O `ctr` tem um teto em 0,15: o pico no limite não segue a tendência das faixas anteriores, e os cliques foram ajustados junto. Para essas 42 campanhas, só sabemos que o CTR real é de pelo menos 15%.

In [24]:
no_teto_eng = train_corrigido["engagement_rate"] == 0.10


def comparar_teto(no_teto):
    resto = ~no_teto
    return {
        "campanhas no teto": no_teto.sum(),
        "plataformas no teto": ", ".join(f"{p} {n}" for p, n in train_corrigido.loc[no_teto, "platform"].value_counts().items()),
        "klike_score mediano (teto)": train_corrigido.loc[no_teto, "klike_score"].median(),
        "klike_score mediano (demais)": train_corrigido.loc[resto, "klike_score"].median(),
        "p (Mann-Whitney)": mannwhitneyu(train_corrigido.loc[no_teto, "klike_score"],
                                         train_corrigido.loc[resto, "klike_score"]).pvalue,
        "% com hook (teto)": train_corrigido.loc[no_teto, "has_hook"].mean() * 100,
        "% com hook (demais)": train_corrigido.loc[resto, "has_hook"].mean() * 100,
    }


pd.DataFrame({
    "ctr = 0,15": comparar_teto(no_teto_ctr),
    "engagement_rate = 0,10": comparar_teto(no_teto_eng),
}).T.style.format({
    "klike_score mediano (teto)": "{:.1f}", "klike_score mediano (demais)": "{:.1f}",
    "p (Mann-Whitney)": "{:.1g}", "% com hook (teto)": "{:.0f}%", "% com hook (demais)": "{:.0f}%",
})

,campanhas no teto,plataformas no teto,klike_score mediano (teto),klike_score mediano (demais),p (Mann-Whitney),% com hook (teto),% com hook (demais)
"ctr = 0,15",42,"Meta 20, TikTok 17, LinkedIn 5",74.4,59.3,2e-07,74%,49%
"engagement_rate = 0,10",7,TikTok 7,57.6,61.2,0.9,43%,52%


O teto do `ctr` não atinge campanhas ao acaso: as campanhas no teto têm `klike_score` bem maior e mais hook. O teto achata justamente as melhores campanhas, então a relação entre CTR e qualidade vai parecer mais fraca do que é.

O teto de `engagement_rate` atinge só o TikTok (7 de 141 campanhas, 5%) e não se associa ao `klike_score`.

In [25]:
piso = train_corrigido["impressions"] == 1000
train_corrigido.loc[piso, ["campaign_id", "platform", "impressions", "clicks", "ctr"]].assign(
    **{"impressões indicadas pelo ctr": (train_corrigido.loc[piso, "clicks"] / train_corrigido.loc[piso, "ctr"]).round()})

,campaign_id,platform,impressions,clicks,ctr,impressões indicadas pelo ctr
371,KLK-0372,Meta,1000,89,0.0899,990.0
40,KLK-0041,Meta,1000,23,0.0234,983.0


In [26]:
pd.crosstab(train_corrigido["video_duration_s"], train_corrigido["platform"]).loc[5:10]

platform,LinkedIn,Meta,TikTok
video_duration_s,,,
5.0,0,5,11
6.0,0,3,7
7.0,0,7,7
8.0,0,6,6
9.0,0,7,13
10.0,2,7,6


Os dois pisos têm efeito pequeno:
- `impressions`: 2 campanhas em exatamente 1.000, com valor original de 983 e 990 (menos de 2% de diferença).
- `video_duration_s`: leve acúmulo em 5 s (16 campanhas, contra 10 em 6 s), quase todas do TikTok. Pode ser um piso, mas a evidência é fraca.

Observações do 4.4:
- `ctr` truncado em 0,15 em 42 campanhas (10,5%), que têm `klike_score` e uso de hook bem acima das demais. Pré-processamento: manter o valor e adicionar a coluna indicadora `ctr_no_teto`.
- `engagement_rate` truncado em 0,10 em 7 campanhas do TikTok, sem associação com o `klike_score`. Pré-processamento: manter o valor e adicionar `engagement_no_teto`.
- Recomendações do engine não podem projetar CTR acima de 15%, fora do que os dados mostram.
- Pisos de `impressions` (1.000) e `video_duration_s` (5 s): apenas registrados, efeito desprezível.